In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import optuna
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, average_precision_score, recall_score
from sklearn.preprocessing import label_binarize
import optuna.visualization as vis
import plotly.io as pio

In [ ]:
# Load the wine dataset
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Adding dummy categorical columns for robustness check
X['binary_cat'] = np.random.choice(['Yes', 'No'], size=len(X))
X['multi_cat'] = np.random.choice(['Low', 'Medium', 'High'], size=len(X))

In [ ]:
# Identify features based on X_train to prevent leakage
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X.select_dtypes(exclude=['number']).columns

binary_features = [col for col in categorical_cols if X[col].nunique() == 2]
multi_cat_features = [col for col in categorical_cols if X[col].nunique() > 2]

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define ColumnTransformer with specific unknown value handling
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('bin_cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), binary_features),
        ('multi_cat', OneHotEncoder(handle_unknown='ignore'), multi_cat_features)
    ]
)

# Create the preprocessing pipeline
preprocessing_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

# Fit on training data and transform both sets
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

In [ ]:
# Create the full pipeline including the model
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Fit the model pipeline
model_pipeline.fit(X_train, y_train)

# Make predictions
y_pred = model_pipeline.predict(X_test)

In [ ]:
# 3. Area Under the Precision-Recall Curve (aucpr)
# Binarize the output for multi-class PR curve calculation
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
y_score = model_pipeline.predict_proba(X_test)

aucpr_list = []
for i in range(len(data.target_names)):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
    aucpr_list.append(auc(recall, precision))

In [ ]:
# Define custom scorers
def aucpr_class(y_true, y_score, labels):
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
    return average_precision_score(y_true_bin[:, labels[0]], y_score[:, labels[0]])

def multiclass_aucpr(y_true, y_score):
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
    return average_precision_score(y_true_bin, y_score, average='macro')

# Scorer for the meta-class: {0} vs {1, 2}
def meta_recall_scorer(y_true, y_pred):
    y_true_binary = (y_true != 0).astype(int)
    y_pred_binary = (y_pred != 0).astype(int)
    return recall_score(y_true_binary, y_pred_binary)

aucpr_scorer = make_scorer(multiclass_aucpr, response_method='predict_proba')
meta_recall = make_scorer(meta_recall_scorer)

# Per-class recall scorers
recall_0 = make_scorer(recall_score, labels=[0], average='macro')
recall_1 = make_scorer(recall_score, labels=[1], average='macro')
recall_2 = make_scorer(recall_score, labels=[2], average='macro')
aucpr_0 = make_scorer(aucpr_class, labels=[0], response_method='predict_proba')
aucpr_1 = make_scorer(aucpr_class, labels=[1], response_method='predict_proba')
aucpr_2 = make_scorer(aucpr_class, labels=[2], response_method='predict_proba')

def objective(trial):
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    max_depth = trial.suggest_int('max_depth', 2, 32)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)

    clf = DecisionTreeClassifier(
        criterion=criterion,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    trial_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])

    # Multi-metric CV
    cv_results = cross_validate(
        trial_pipeline, X_train, y_train, cv=5,
        scoring={
            'bal_acc': 'balanced_accuracy',
            'f1_macro': 'f1_macro',
            'aucpr': aucpr_scorer,
            'meta_rec': meta_recall,
            'rec0': recall_0,
            'rec1': recall_1,
            'rec2': recall_2,
            'aucpr0': aucpr_0,
            'aucpr1': aucpr_1,
            'aucpr2': aucpr_2
        }
    )

    # Log all metrics as user attributes
    trial.set_user_attr("f1_macro", cv_results['test_f1_macro'].mean())
    trial.set_user_attr("aucpr", cv_results['test_aucpr'].mean())
    trial.set_user_attr("meta_non_zero_recall", cv_results['test_meta_rec'].mean())
    trial.set_user_attr("recall_class_0", cv_results['test_rec0'].mean())
    trial.set_user_attr("recall_class_1", cv_results['test_rec1'].mean())
    trial.set_user_attr("recall_class_2", cv_results['test_rec2'].mean())
    trial.set_user_attr("aucpr_class_0", cv_results['test_aucpr0'].mean())
    trial.set_user_attr("aucpr_class_1", cv_results['test_aucpr1'].mean())
    trial.set_user_attr("aucpr_class_2", cv_results['test_aucpr2'].mean())

    return cv_results['test_bal_acc'].mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# Refit best model
best_clf = DecisionTreeClassifier(**study.best_params, random_state=42)
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', best_clf)])
model_pipeline.fit(X_train, y_train)

In [ ]:
# 1. Optimization History
vis.plot_optimization_history(study).show()

# 2. Hyperparameter Importances
vis.plot_param_importances(study).show()

# 3. Parallel Coordinate Plot (View relationships between params and objective)
vis.plot_parallel_coordinate(study).show()

# 4. Slice Plot (View parameter distribution against objective)
vis.plot_slice(study).show()

In [ ]:
vis.plot_edf(study).show()

In [ ]:
vis.plot_contour(study, params=['max_depth', 'min_samples_split']).show()

In [ ]:
vis.plot_rank(study).show()

In [ ]:
# Display the full trials results as a DataFrame for analysis
df_results = study.trials_dataframe()

# Sort by value to see the best trials at the top
display(df_results.sort_values(by='value', ascending=False).head(10))